<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day18_practice2_%ED%8A%B8%EB%9E%9C%EC%8A%A4%ED%8F%AC%EB%A8%B8_%EB%B8%94%EB%A1%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from torch._C import device
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import urllib.request, os, math
from collections import Counter # Counter=단어 빈도 세기

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# 셀 1. 쪼개고, 각자 보고, 합친다
class MultiHeadAttention(nn.Module):
  def __init__(self, dim, n_heads): # dim=64, n_head=8
    super().__init__()
    assert dim % n_heads == 0 # 64/8 == 0
    self.n_heads = n_heads
    self.d_head = dim // n_heads # d_head=8 헤드 하나가 맡는 차원수

    self.W_q = nn.Linear(dim, dim, bias=False) # 각 헤드마다 Q,K,V를 만들 학습 행렬
    self.W_k = nn.Linear(dim, dim, bias=False)
    self.W_v = nn.Linear(dim, dim, bias=False)
    self.W_o = nn.Linear(dim, dim, bias=False) # 헤드들 결과를 합친 뒤 섞는 출력층

  def forward(self, x, mask=None):
    B, L, D = x.shape   # x1(2,10,64) 배치, 단어수(Length), 벡터차원
    H, dh = self.n_heads, self.d_head # H=8 헤드수, dh=8 각 헤드 차원수

    # 1. 쪼개기 x1(2,10,64) → (2,10,8,8) → (2,8,10,8)
    Q = self.W_q(x).reshape(B, L, H, dh).transpose(1,2) # (2,8,10,8) 배치, 멀티헤드수, 단어길이, 차원
    K = self.W_k(x).reshape(B, L, H, dh).transpose(1,2) # (2,8,10,8)
    V = self.W_v(x).reshape(B, L, H, dh).transpose(1,2) # (2,8,10,8)

    # 2. 각자 보기: 셀프어텐션
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dh) # [(2,8,10,10)] (B, H, QL, KL) 관련도 점수
    if mask is not None:
      scores = scores.masked_fill(mask, -1e9) # masked_fill(조건, 값) 조건이 True인 위치를 값으로 채운다, -10억(-무한대)이 softmax를 통과
    attn = F.softmax(scores, dim=-1) # softmax에서 dim=-1 축을 따라 정규화, (축은 그대로 남아있고, 그 축의 값들을 합=1로), (2,8,100,10K)
    self.attn_map = attn # 시각화용 저장, (2,8,10,10)
    out = attn @ V # (2,8,10,8) B, H, L, dh

    # 3. 이어붙이기 : 헤드 8개(각 8차원)를 다시 원래 64차원 한 줄로 이어붙여 입력과 같은 모양 복원 (2,10,64)
    out = out.transpose(1,2).reshape(B, L, D) #(2,8,10,8) → (2,10,8,8) → (2,10,64)

    # 4. 섞기 : 헤드 8개 결과를 어떻게 조합할지도 학습 (W_o)
    return self.W_o(out) # (2,10,64)


In [9]:
# 셀 1. 트랜스포머 블록
#   x ──┬─ MultiHead ──(+x)── LayerNorm ──┬─ FFN ──(+x)── LayerNorm → 출력
#       └────── 잔차 ↑──────┘             └───── 잔차 ↑─────┘

# 1x(잔차)   : 스킵 커넥션
# LayerNors : BatchNors, 배치가 아니라 '단어 벡터별' 표준화
# FFN (Free-Forward Network) : MLP (dim → 4xdim → dim, 단어별로 독립 적용) 각 단어가 받은 정보를 '혼자 가공'

class TransformerBlock(nn.Module):
  def __init__(self, dim=64, n_heads=8):
    super().__init__()
    self.attn = MultiHeadAttention(dim, n_heads) # 단어끼리 정보 교환
    self.norm1 = nn.LayerNorm(dim)
    self.ffn = nn.Sequential( # FFN
        nn.Linear(dim, dim * 4), nn.ReLU(), nn.ReLU(), nn.Linear(dim * 4, dim)

    )
    self.norm2 = nn.LayerNorm(dim)

  def forward(self, x, mask=None): # x1(2,10,64)
    x = self.norm1(x + self.attn(x, mask)) # + x 잔차
    x = self.norm2( x + self.ffn(x))
    return x #(2,10,64) 입력과 똑같은 모양

In [10]:
block = TransformerBlock()
x = torch.rand(2, 10, 64)
print("블록 통과:", tuple(x.shape), "->", tuple(block(x).shape)) # 입력과 출력의 모양이 같다. 블록을 수십개 쌓아도 되는 이유.
print(f"블록 하나 파라미터: {sum(p.numel() for p in block.parameters())/1e3:.0f}K")


블록 통과: (2, 10, 64) -> (2, 10, 64)
블록 하나 파라미터: 50K


In [11]:
# 셀 2. PyTorch 공식 구현과 파라미터 대조
official = nn.TransformerEncoderLayer(d_model=64, nhead=8, dim_feedforward=258, batch_first=True)
n_ours = sum(p.numel() for p in block.parameters())
n_official = sum(p.numel() for p in official.parameters())
print(f"\n파라미터: 우리 블록 {n_ours:,} vs nn.TransformerEncoderLayer {n_official:,}") # 차이 256 = 공식은 bias 를 더 씀



파라미터: 우리 블록 49,728 vs nn.TransformerEncoderLayer 50,242


In [13]:
# 최신 LLM 스타일로 레이어 쌓기
num_layer = nn.TransformerEncoderLayer(d_model=64, nhead=8, batch_first=True)
num_layers = 32
transformer_encoder = nn.TransformerEncoder(num_layer, num_layers=num_layers) # 32층으로 찾는다. (중간크기 LLM 40층)


In [15]:
# 셀 3. 마스크드 어텐션 - GPT 의 '커닝 방지'
# 어텐션은 모든 단어를 본다.
# GPT의 학습: '다음 단어 맞히기' 미래 단어를 가리는 마스크가 필요

# 인과(causal) 마스크 만들기
L =  6
causal_mask = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
print("\n인과(causal) 마스크 - True(■) 자리는 못 본다:")
for i in range(L):
  print(" ", "".join("■" if causal_mask[i, j] else "·" for j in range(L)) + f" ← {i+1}번째 단어는 {i+1}번까지만")
out = block(x[:, :L, :], mask=causal_mask)
print("마스크 장착 블록 통과:", tuple(out.shape), "- 각 단어는 '과거만' 본다")
# 인코더 (마스크 없음): 문장 전체를 양방향으로 이해 - BERT 계열, 분류 · 검색
# 디코더 (인과 마스크): 과거만 보고 다음을 생성 - GPT · Claude LLM 계열




인과(causal) 마스크 - True(■) 자리는 못 본다:
  ·■■■■■ ← 1번째 단어는 1번까지만
  ··■■■■ ← 2번째 단어는 2번까지만
  ···■■■ ← 3번째 단어는 3번까지만
  ····■■ ← 4번째 단어는 4번까지만
  ·····■ ← 5번째 단어는 5번까지만
  ······ ← 6번째 단어는 6번까지만
마스크 장착 블록 통과: (2, 6, 64) - 각 단어는 '과거만' 본다


In [16]:
# 셀 4. 잔차의 힘 (+ x) - 블록을 깊이 쌓아서 확인
def first_grad(n_blocks, use_residual):
  torch.manual_seed(0)
  blocks = nn.ModuleList(TransformerBlock() for _ in range(n_blocks))
  torch.manual_seed(1)
  x = torch.rand(1, 10, 64, requires_grad=True)
  h = x
  for b in blocks:
    if use_residual:
      h = b.norm1(h + b.attn(h))
      h = b.norm2(h + b.ffn(h))
    else:
      h = b.norm1(b.attn(h))
      h = b.norm2(b.ffn(h))
  h.sum().backward()
  return x.grad.abs().mean().item()

print("\n[입력의 기울기 - 블록 깊이별]")
print(f"{'깊이':>6s} | {'잔차 제거':>12s} | {'잔차 있음':>12s}")
for n in [12, 24, 48]: # 블록 층을 쌓는다
  print(f"{n:>6d} | {first_grad(n, False):>12.2e} | {first_grad(n, True):>12.2e}")


[입력의 기울기 - 블록 깊이별]
    깊이 |        잔차 제거 |        잔차 있음
    12 |     3.75e-09 |     1.45e-08
    24 |     1.08e-09 |     7.64e-09
    48 |     2.63e-11 |     1.48e-08
